In [ ]:
# Mixing curve. Goes in a fresh notebook, mixing.ipynb.
#
# Roughly 45 training runs. Human-only at small budgets is fast; the mixed arms carry
# all 18,482 synthetic instances every time, so budget ~40 minutes total.
#
# Nothing is generated. This runs entirely on data that already exists.


# ===========================================================================
# Cell 1: setup
# ===========================================================================
import importlib
import ddi.mixing, ddi.train
for m in (ddi.mixing, ddi.train):
    importlib.reload(m)

from ddi.data import build_human
from ddi.manifest import load_dataset
from ddi.train import train_and_eval
from ddi.experiment import log_run
from ddi import mixing

V14_ID = "20260807-123340-ff79db"

train, dev, val = build_human()
v14, v14_man = load_dataset(V14_ID)

n_human_sents = len({r["sent_id"] for r in train})
n_v14_sents = len({r["sent_id"] for r in v14})
print(f"human {len(train):>6} instances / {n_human_sents} sentences, "
      f"pos rate {mixing.positive_rate(train):.3f}")
print(f"v14   {len(v14):>6} instances / {n_v14_sents} sentences, "
      f"pos rate {mixing.positive_rate(v14):.3f}")

BASE = {"model_name": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
        "epochs": 3, "lr": 2e-5, "batch_size": 32, "max_length": 256,
        "neg_ratio": None, "render_mode": "markers", "synth_id": V14_ID}



In [ ]:

# ===========================================================================
# Cell 2: sanity check the subsampler before spending 40 minutes on it
# ===========================================================================
for n in [100, 500, None]:
    sub = mixing.subsample_by_sentence(train, n, seed=0)
    print(f"{str(n):>5} sentences -> {len(sub):>6} instances, "
          f"{len({r['sent_id'] for r in sub}):>5} sentences, "
          f"pos rate {mixing.positive_rate(sub):.3f}")

# instances per sentence should stay near 11.0 at every budget. If it does not, the
# subsample has caught or missed the eight monster sentences and the budget axis is
# not measuring what it says.



In [ ]:
from collections import Counter
per_sent = Counter(r["sent_id"] for r in train)
monsters = {s for s, n in per_sent.items() if n >= 190}   # 20+ entities
train_f = [r for r in train if r["sent_id"] not in monsters]
print(f"{len(monsters)} sentences removed, {len(train_f)} instances left")
for n in [100, 500, None]:
    sub = mixing.subsample_by_sentence(train_f, n, seed=0)
    print(f"{str(n):>5} -> {len(sub):>6} inst, pos {mixing.positive_rate(sub):.3f}")

In [ ]:
# ===========================================================================
# Cell 3: the curve, uncontrolled. ~40 min.
# ===========================================================================
rows = mixing.run_curve(train, v14, dev, train_and_eval, BASE,
                        match_balance=False, log=log_run)

summary, detail = mixing.summarise(rows)
print(summary.to_string(index=False))

import json
from pathlib import Path
Path("runs/mixing-v14.json").write_text(json.dumps(rows, indent=1))

In [ ]:
# ===========================================================================
# Cell 4: plot
# ===========================================================================
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame(rows)
g = df.groupby(["n_sentences", "arm"]).f1.agg(["mean", "std"]).reset_index()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

for arm, style in [("human", "o-"), ("human+synth", "s-")]:
    s = g[g.arm == arm]
    ax[0].errorbar(s.n_sentences, s["mean"], yerr=s["std"], fmt=style, label=arm,
                   capsize=3)
ax[0].axhline(0.379, ls=":", c="grey", label="v14 alone")
ax[0].set_xscale("log")
ax[0].set_xlabel("human sentences")
ax[0].set_ylabel("dev micro-F1, positive classes")
ax[0].legend()

ax[1].axhline(0, c="k", lw=0.8)
ax[1].errorbar(summary.n_sentences, summary.delta_f1, yerr=summary.pooled_sd,
               fmt="o-", capsize=3)
ax[1].set_xscale("log")
ax[1].set_xlabel("human sentences")
ax[1].set_ylabel("F1 gained by adding synthetic")

plt.tight_layout()
plt.savefig("figures/mixing-curve-v14.png", dpi=150)
plt.show()

In [ ]:
# ===========================================================================
# Cell 5: balance control, crossover region only.
# Mixing shifts the positive rate from 0.096 to somewhere near 0.15. If the gain is
# really a class-balance effect, matching the rate will remove it.
# Pick the two or three budgets either side of where delta_f1 crosses zero.
# ===========================================================================
CROSSOVER = [500, 1000, 2500]      # set from cell 3's output

rows_bal = mixing.run_curve(train, v14, dev, train_and_eval, BASE,
                            budgets=CROSSOVER, match_balance=True, log=log_run)
summary_bal, _ = mixing.summarise(rows_bal)
print(summary_bal.to_string(index=False))

print("\nuncontrolled, same budgets:")
print(summary[summary.n_sentences.isin(CROSSOVER)].to_string(index=False))